In [ ]:
import tensorflow as tf
from keras.layers import Conv2D, Input, ZeroPadding2D, BatchNormalization, Activation, MaxPooling2D, Flatten, Dense
from keras.models import Model, load_model
from keras.callbacks import TensorBoard, ModelCheckpoint
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score,recall_score,precision_score,ConfusionMatrixDisplay,roc_curve,confusion_matrix,auc
from sklearn.utils import shuffle
import cv2
import imutils
import numpy as np
import matplotlib.pyplot as plt
import time
import random
import os
import nibabel as nib

In [ ]:
from google.colab import drive
drive.mount('/content/drive',force_remount=True)

Mounted at /content/drive


In [ ]:
def read_nifti_file(filepath):
    """Read and load volume"""
    # Read file
    scan = nib.load(filepath)
    # Get raw data
    scan = scan.get_fdata()
    return scan

In [ ]:
DEFAULT_RANDOM_SEED = 1530

def seedBasic(seed=DEFAULT_RANDOM_SEED):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    
# tensorflow random seed 
import tensorflow as tf 
def seedTF(seed=DEFAULT_RANDOM_SEED):
    tf.random.set_seed(seed)
    
# torch random seed
import torch
def seedTorch(seed=DEFAULT_RANDOM_SEED):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
      
# basic + tensorflow + torch 
def seedEverything(seed=DEFAULT_RANDOM_SEED):
    seedBasic(seed)
    seedTF(seed)
    seedTorch(seed)


seedEverything()

In [ ]:
# path = '/content/drive/My Drive/COBRE/YES/0040000/session_1/anat_1/mprage.nii.gz'
# img_data = read_nifti_file(path)

In [ ]:
# print(type(img_data))  # it's a numpy array!
# print(img_data.shape)

# mid_vox = img_data[94:97, 126:129, 126:129]
# print(mid_vox)

In [ ]:
# mid_slice_x = img_data[:, :, 138]
# print(mid_slice_x.shape)
# plt.imshow(mid_slice_x.T, cmap='gray', origin='lower')
# plt.xlabel('First axis')
# plt.ylabel('Second axis')
# plt.colorbar(label='Signal intensity')
# plt.show()

In [ ]:
# n_slice = img_data.shape[1]
# step_size = n_slice // 36
# plot_range = 36 * step_size
# start_stop = int((n_slice - plot_range) / 2)

# fig, axs = plt.subplots(6, 6, figsize=[10, 10])

# for idx, img in enumerate(range(start_stop, plot_range, step_size)):
#     axs.flat[idx].imshow(img_data[:, :, img], 
#                          cmap='gray')
#     axs.flat[idx].axis('off')
        

# plt.tight_layout()
# plt.show()
# plt.savefig('brain_image.png')





In [ ]:
# directory  = '/content/drive/My Drive/COBRE/YES'
# for filename in os.listdir(directory):

#   new_session_file = directory+'/'+filename+'/session_1/anat_1/mprage.nii.gz'
#   img_data = read_nifti_file(new_session_file)

#   for i in range(1,8):
#     test1 = img_data[:, :, 149+i]
#     cv2.imwrite(directory+'/'+filename+'/test'+'{}'.format(i)+'.jpg', test1)




In [ ]:
# directory  = '/content/drive/My Drive/COBRE/NO'
# for filename in os.listdir(directory):

#   new_session_file = directory+'/'+filename+'/session_1/anat_1/mprage.nii.gz'
#   img_data = read_nifti_file(new_session_file)

#   for i in range(1,8):
#     test1 = img_data[:, :, 149+i]
#     cv2.imwrite(directory+'/'+filename+'/test'+'{}'.format(i)+'.jpg', test1)




In [ ]:
def preprocessing_image(img_data):

  ret,thresh1 = cv2.threshold(img_data,127,255,cv2.THRESH_BINARY)
  blur = cv2.blur(thresh1,(5,5),0)

  return blur


In [ ]:
def split_data(X, y, test_size=0.2):
       
    """
    Splits data into training, development and test sets.
    Arguments:
        X: A numpy array with shape = (#_examples, image_width, image_height, #_channels)
        y: A numpy array with shape = (#_examples, 1)
    Returns:
        X_train: A numpy array with shape = (#_train_examples, image_width, image_height, #_channels)
        y_train: A numpy array with shape = (#_train_examples, 1)
        X_val: A numpy array with shape = (#_val_examples, image_width, image_height, #_channels)
        y_val: A numpy array with shape = (#_val_examples, 1)
        X_test: A numpy array with shape = (#_test_examples, image_width, image_height, #_channels)
        y_test: A numpy array with shape = (#_test_examples, 1)
    """
    
    X_train, X_test_val, y_train, y_test_val = train_test_split(X, y, test_size=test_size,stratify = y)
    X_test, X_val, y_test, y_val = train_test_split(X_test_val, y_test_val, test_size=0.5,stratify = y_test_val)
    
    return X_train, y_train, X_val, y_val, X_test, y_test


In [ ]:
X_file = []
y_file = []
directory  = '/content/drive/My Drive/COBRE/YES'
for filename in os.listdir(directory):
  X_file.append(filename)
  y_file.append([1])

directory  = '/content/drive/My Drive/COBRE/NO'
for filename in os.listdir(directory):
  X_file.append(filename)
  y_file.append([0])

X_file, y_file = shuffle(X_file, y_file)
X_train_file, y_train_file, X_val_file, y_val_file, X_test_file, y_test_file = split_data(X_file, y_file, test_size=0.3)



In [ ]:
def multiplying_data(x_var , y_var):

  X_train=[]
  y_train=[]
  for ind in range(len(x_var)):

    filename = x_var[ind]
    output = y_var[ind]
    if output == [0]:
      directory = '/content/drive/My Drive/COBRE/'+'NO'
    else:
      directory = '/content/drive/My Drive/COBRE/'+'YES'

    for i in range(1,8):
      path = directory+'/'+filename+'/test'+'{}'.format(i)+'.jpg'
      image = cv2.imread(path)
      image = preprocessing_image(image)
      X_train.append(image)
      y_train.append(output)

  X_train = np.array(X_train)
  y_train = np.array(y_train) 

  return X_train,y_train


In [ ]:
def normal_test_data(x_var,y_var):
  X_train=[]
  y_train=[]
  for ind in range(len(x_var)):
    filename = x_var[ind]
    output = y_var[ind]
    if output == [0]:
      directory = '/content/drive/My Drive/COBRE/'+'NO'
    else:
      directory = '/content/drive/My Drive/COBRE/'+'YES'

    path = directory+'/'+filename+'/test1.jpg'
    image = cv2.imread(path)
    image = preprocessing_image(image)
    X_train.append(image)
    y_train.append(output)

  X_train = np.array(X_train)
  y_train = np.array(y_train) 

  return X_train,y_train



In [ ]:
X_train , y_train = multiplying_data(X_train_file,y_train_file)
X_val , y_val  = multiplying_data(X_val_file,y_val_file)
X_test , y_test = normal_test_data(X_test_file,y_test_file)

In [ ]:
# def plot_sample_images(X, y, n=50):
#     """
#     Plots n sample images for both values of y (labels).
#     Arguments:
#         X: A numpy array with shape = (#_examples, image_width, image_height, #_channels)
#         y: A numpy array with shape = (#_examples, 1)
#     """
    
#     for label in [0,1]:
#         # grab the first n images with the corresponding y values equal to label
#         images = X[np.argwhere(y == label)]
#         n_images = images[:n]
        
#         columns_n = 10
#         rows_n = int(n/ columns_n)

#         plt.figure(figsize=(20, 10))
        
#         i = 1 # current plot        
#         for image in n_images:
#             plt.subplot(rows_n, columns_n, i)
#             plt.imshow(image[0])
            
#             # remove ticks
#             plt.tick_params(axis='both', which='both', 
#                             top=False, bottom=False, left=False, right=False,
#                            labelbottom=False, labeltop=False, labelleft=False, labelright=False)
            
#             i += 1
        
#         label_to_str = lambda label: "Yes" if label == 1 else "No"
#         plt.suptitle(f"Schizophrenia: {label_to_str(label)}")
#         plt.show()

In [ ]:
# plot_sample_images(X, y)

In [ ]:
print(X_train.shape)

(714, 192, 256, 3)


In [ ]:
from keras.applications.vgg19 import VGG19
base_model = VGG19(weights="imagenet", include_top=False, input_shape=X_train[0].shape)
base_model.trainable = False

80134624/80134624 [==============================] - 5s 0us/step


In [ ]:
from tensorflow.keras import layers, models

flatten_layer = layers.Flatten()
dropout_layer = layers.Dropout(0.3)
dense_layer_1 = layers.Dense(128, activation='relu')
bn_layer = BatchNormalization(axis = 1, name = 'bn0')
dense_layer_2 = layers.Dense(64, activation='relu')
prediction_layer = layers.Dense(1, activation='sigmoid')
dense_layer_3 = layers.Dense(32, activation='relu')


model = models.Sequential([
    base_model,
    flatten_layer,
    dropout_layer,
    dense_layer_1,
    dropout_layer,
    dense_layer_2,
    dropout_layer,
    dense_layer_3,
    prediction_layer
])

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy'],
)
es = EarlyStopping(monitor='val_accuracy', mode='max', patience=20,  restore_best_weights=True)

In [ ]:
model.fit(x=X_train, y=y_train, batch_size=32, epochs=100, validation_data=(X_val, y_val),callbacks=[es])

In [ ]:
history = model.history.history
for key in history.keys():
    print(key)

In [ ]:
def plot_metrics(history):
    
    train_loss = history['loss']
    val_loss = history['val_loss']
    train_acc = history['accuracy']
    val_acc = history['val_accuracy']
    
    # Loss
    plt.figure()
    plt.plot(train_loss, label='Training Loss')
    plt.plot(val_loss, label='Validation Loss')
    plt.title('Loss')
    plt.legend()
    plt.show()
    
    # Accuracy
    plt.figure()
    plt.plot(train_acc, label='Training Accuracy')
    plt.plot(val_acc, label='Validation Accuracy')
    plt.title('Accuracy')
    plt.legend()
    plt.show()

plot_metrics(history) 

In [ ]:
loss, acc = model.evaluate(x=X_test, y=y_test)

In [ ]:
print (f"Test Loss = {loss}")
print (f"Test Accuracy = {acc}")

In [ ]:
def compute_f1_score(y_true, prob):
    # convert the vector of probabilities to a target vector
    y_pred = np.where(prob > 0.5, 1, 0)
    
    score = f1_score(y_true, y_pred)
    
    return score

def compute_recall_score(y_true,prob):

    y_pred = np.where(prob > 0.5, 1, 0)
    
    score = recall_score(y_true, y_pred)
    
    return score


def compute_precision_score(y_true,prob):

    y_pred = np.where(prob > 0.5, 1, 0)
    
    score = recall_score(y_true, y_pred)
    
    return score

# def compute_confusion_matrix(y_true,prob):

#     y_pred = np.where(prob > 0.5, 1, 0)
#     return confusion_matrix(y_true,y_pred,labels=[0,1])

def compute_roc_curve(y_true,prob):

    y_pred = np.where(prob > 0.5, 1, 0)
    
    fpr, tpr, threshold  = roc_curve(y_true, y_pred)
    
    return fpr, tpr, threshold 


In [ ]:
y_test_prob = model.predict(X_test)
f1score = compute_f1_score(y_test, y_test_prob)
recall = compute_recall_score(y_test,y_test_prob)
precision = compute_precision_score(y_test,y_test_prob)
# confusion_mat = compute_confusion_matrix(y_test,y_test_prob)
# disp = ConfusionMatrixDisplay(confusion_matrix = confusion_mat,display_labels = [0,1])
# disp.plot()

fpr, tpr, threshold = compute_roc_curve(y_test, y_test_prob)
roc_auc = auc(fpr, tpr)

plt.plot(fpr,tpr,label="data 1, auc="+str(auc))
plt.legend(loc=4)
plt.show()

print(f"F1 score: {f1score}")
print(f"Recall: {recall}")
print(f"Precision: {precision}")